In [1]:
from transformers import pipeline

# load the zero-shot classifier (first run downloads the model — may take a minute)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["opportunity", "risk", "trend"]

text = "Lufthansa faces pilot strikes and rising fuel costs threatening its profits."
result = classifier(text, candidate_labels=labels)

print(result)

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'sequence': 'Lufthansa faces pilot strikes and rising fuel costs threatening its profits.', 'labels': ['risk', 'trend', 'opportunity'], 'scores': [0.9322348237037659, 0.05446822941303253, 0.013296927325427532]}


In [2]:
top_label = result["labels"][0]    # 'risk'  (first = highest score)
top_score = result["scores"][0]    # 0.93    (its confidence)

print("Category:", top_label)
print("Confidence:", round(top_score, 2))

Category: risk
Confidence: 0.93


In [3]:
import json
documents = json.load(open("lufthansa_data.json", encoding="utf-8"))

labels = ["opportunity", "risk", "trend"]

# test on the first 5 real docs
for d in documents[:5]:
    result = classifier(d["text"], candidate_labels=labels)
    cat   = result["labels"][0]
    score = result["scores"][0]
    print(f"[{cat}] ({score:.2f})  {d['text'][:90]}")
    print()

[trend] (0.44)  Financial reports - Lufthansa Group Investor Relations. Lufthansa Group publishes its 1st 

[trend] (0.56)  Financial reports & publications - Lufthansa Group Investor Relations. The Lufthansa Group

[trend] (0.53)  Financial Data - Lufthansa Technik. Lufthansa Technik AG again generated earnings of more 

[trend] (0.68)  Lufthansa narrows losses in first quarter as demand offsets rising fuel . Lufthansa Group 

[trend] (0.52)  Lufthansa Group Posts Record Revenue, Profit Surge. The Lufthansa Group has reported its s



In [4]:
sentiment_pipe = pipeline("sentiment-analysis",
                          model="cardiffnlp/twitter-roberta-base-sentiment-latest")

#3-class sentiment (negative / neutral / positive)
from collections import Counter

counts = Counter()
for d in documents:
    label = sentiment_pipe(d["text"][:512])[0]["label"]
    d["sentiment"] = label          # store it on the doc (reuse later — no re-running)
    counts[label] += 1

print(counts)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Counter({'neutral': 109, 'positive': 66, 'negative': 20})


In [6]:
labels = ["opportunity", "risk", "trend"]

for d in documents:
    d["category"] = classifier(d["text"], candidate_labels=labels)["labels"][0]

print("Done classifying")

# see the category spread
from collections import Counter
print(Counter(d["category"] for d in documents))

Done classifying
Counter({'trend': 104, 'risk': 79, 'opportunity': 12})


In [7]:
import json
json.dump(documents,
          open("lufthansa_labeled.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Saved", len(documents), "labeled docs")

Saved 195 labeled docs
